# keybed segmentation on a pretrained backbone

Same mask output the geometry already fits at 2.0 px; the backbone is ImageNet pretrained so
invariance comes from the weights rather than from renders.


In [ ]:
import json
import shutil
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import torch

SRC = next(Path("/kaggle/input").rglob("segnet2.py")).parent
CORPUS = next(Path("/kaggle/input").rglob("corners.json")).parent
PACKAGE = Path("/kaggle/working/kvt")
OUT = Path("/kaggle/working/keybed_seg2.pt")

shutil.rmtree(PACKAGE, ignore_errors=True)
shutil.copytree(SRC, PACKAGE)
sys.path.insert(0, str(PACKAGE.parent))

from kvt.segnet2 import SEG2_INPUT_SIZE, KeybedSegNet2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpus = torch.cuda.device_count()
BATCH = 48
EPOCHS = 25
MASK_SIZE = 144
RASTER = MASK_SIZE * 4
print(f"{gpus} gpu(s), batch {BATCH}, input {SEG2_INPUT_SIZE}")

In [ ]:
corners_by_stem = json.loads((CORPUS / "corners.json").read_text())
stems = sorted(corners_by_stem)


def load(stem):
    image = cv2.imread(str(CORPUS / "frames" / f"{stem}.png"), cv2.IMREAD_GRAYSCALE)
    if image is None:
        return None
    rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    frame = cv2.resize(rgb, (SEG2_INPUT_SIZE, SEG2_INPUT_SIZE), interpolation=cv2.INTER_AREA)
    quad = np.array(corners_by_stem[stem], dtype=np.float64) * RASTER
    big = np.zeros((RASTER, RASTER), dtype=np.uint8)
    cv2.fillPoly(big, [quad.astype(np.int32)], 255)
    mask = cv2.resize(big, (MASK_SIZE, MASK_SIZE), interpolation=cv2.INTER_AREA)
    return frame, mask


with ThreadPoolExecutor(max_workers=8) as workers:
    pairs = [p for p in workers.map(load, stems) if p is not None]
images = np.stack([p[0] for p in pairs])
masks = np.stack([p[1] for p in pairs])
print(f"{len(images)} frames  {images.shape}  masks {masks.shape}")


In [ ]:
import gc
import math

from torch import nn
from torch.nn import functional as F

rng = np.random.default_rng(0)
order = rng.permutation(len(images))
cut = int(len(order) * 0.08)
val, train = order[:cut], order[cut:]


# the whole corpus lives as one uint8 tensor and normalising happens on the gpu, because the
# numpy float32 copies this used to make per batch are what kept killing the kernel
store = torch.from_numpy(images)
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def batch(index, jitter):
    raw = store[torch.from_numpy(np.ascontiguousarray(index))].to(device, non_blocking=True)
    out = raw.permute(0, 3, 1, 2).float().div_(255.0)
    if jitter:
        n = out.shape[0]
        gain = torch.empty(n, 1, 1, 1, device=device).uniform_(0.7, 1.4)
        bias = torch.empty(n, 1, 1, 1, device=device).uniform_(-0.16, 0.16)
        out = (out * gain + bias).clamp_(0.0, 1.0)
    return out.sub_(MEAN).div_(STD)


def truth(index):
    return torch.from_numpy(masks[index].astype(np.float32) / 255.0).unsqueeze(1).to(device)


def iou(logits, target):
    predicted = torch.sigmoid(logits) > 0.5
    actual = target > 0.5
    return float((predicted & actual).sum()), float((predicted | actual).sum())


torch.manual_seed(0)
model = KeybedSegNet2().to(device)
# every run that died was replicating this tiny model across both gpus each step;
# one card is plenty for 1.6M parameters and it is the only variable left
parallel = model
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
steps = math.ceil(len(train) / BATCH) * EPOCHS
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=1e-6)

best = 0.0
for epoch in range(EPOCHS):
    parallel.train()
    shuffled = rng.permutation(train)
    for start in range(0, len(shuffled), BATCH):
        pick = shuffled[start : start + BATCH]
        if len(pick) < 2:
            continue
        optimizer.zero_grad(set_to_none=True)
        loss = F.binary_cross_entropy_with_logits(parallel(batch(pick, True)), truth(pick))
        loss.backward()
        optimizer.step()
        scheduler.step()
    model.eval()
    inter = union = 0.0
    with torch.no_grad():
        for start in range(0, len(val), BATCH):
            pick = val[start : start + BATCH]
            a, b = iou(model(batch(pick, False)), truth(pick))
            inter += a
            union += b
    score = inter / max(union, 1.0)
    gc.collect()
    torch.cuda.empty_cache()
    # synthetic iou rises while real transfer falls, so the checkpoint cannot be chosen here;
    # every epoch is kept and the pick happens against real frames on the workstation
    torch.save(model.state_dict(), OUT.with_name(f"seg2-e{epoch + 1:02d}.pt"))
    best = max(best, score)
    print(f"epoch {epoch + 1}/{EPOCHS} val_iou {score:.4f}", flush=True)

print(f"saved {OUT} val_iou {best:.4f}")
